In [ ]:
import os
import copy
import glob
import warnings # hide the warnings
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd                          
import torch
import torch.nn as nn


from scipy import sparse
from scipy.stats import lognorm
from scipy.stats import genextreme
from scipy.interpolate import griddata

from matplotlib.colors import BoundaryNorm
from matplotlib.colors import ListedColormap
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt 
from cartopy.io import shapereader   
import cartopy.crs as ccrs

from climada.hazard import Hazard
from climada.hazard import Centroids
from climada.hazard import TCTracks
from climada.hazard import TropCyclone
from climada.util.plot import plot_from_gdf

warnings.filterwarnings("ignore")

In [ ]:
###############################################################################################################################################
# below we use WRF (WRF has no synthetic)
###############################################################################################################################################

In [ ]:
files_list = sorted(glob.glob("/lfs/home/yanlan/climada/tc-risk/PGW4K.v230926/wrfout_d01_*_48.nc")) # a list of all historical file names

In [ ]:
years = [int(os.path.basename(f).split("_")[2][:4]) for f in files_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
###############################################################################################################################################
# create the WRF hazard object
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of xarray.DataArray (499*549)

    centroids = Centroids( lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                           lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()] ) # centroids are the "true" points in mask

###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
n_ev = len(files_list)
###############################################################################################################################################
event_id = np.array(range(1, n_ev + 1), dtype=int)
###############################################################################################################################################
intensities_list = []
for f in files_list:
    with xr.open_dataset(f) as ref:
        u = ref["U10"].values[:, mask.values] 
        v = ref["V10"].values[:, mask.values] 
        
        max_wind = np.sqrt(u**2 + v**2).max(axis=0) # max on the time dimension
        intensities_list.append(max_wind)

intensity = sparse.csr_matrix(np.vstack(intensities_list))
###############################################################################################################################################
# frequency -> equal weighting
frequency = np.array([ (n_ev / (max(years) - min(years) + 1)) / n_ev  ] * n_ev)
###############################################################################################################################################
fraction = intensity.copy()
fraction.data.fill(1)
###############################################################################################################################################
event_name = [f.split("_")[2] for f in files_list] # ['wrfout', 'd01', '202002MEKKHALA', '48.nc']
###############################################################################################################################################
# date = np.array([  ])
###############################################################################################################################################
# orig = np.ones(n_ev, bool)

In [ ]:
haz = Hazard(intensity=intensity,
             fraction=fraction,
             centroids=centroids,  
             units=units,
             event_id=event_id,
             frequency=frequency,
             event_name=event_name)

In [ ]:
haz.check()

In [ ]:
haz.plot_intensity(0, vmin=0, vmax=70)

haz.plot_intensity("201503SOUDELOR")

haz.plot_intensity(195, vmin=0, vmax=35)
haz.plot_intensity(196, vmin=0, vmax=35)
haz.plot_intensity(197, vmin=0, vmax=35)


In [ ]:
###############################################################################################################################################
# below we use TC Track data (txt files)
###############################################################################################################################################

In [ ]:
tracks_list = sorted(glob.glob("/lfs/home/yanlan/climada/tc-risk/tc_track_data/TCtrack.slp.4C_*_48.txt"))

In [ ]:
years = [int(os.path.basename(f).split("_")[1][:4]) for f in tracks_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
def compute_category_ms(vmax_ms):  # saffir_simpson_category
    if vmax_ms < 17.49:   # 34 kt
        return -1
    if vmax_ms < 32.92:   # 64 kt
        return 0
    if vmax_ms < 42.70:   # 83 kt
        return 1
    if vmax_ms < 49.39:   # 96 kt
        return 2
    if vmax_ms < 58.13:   # 113 kt
        return 3
    if vmax_ms < 70.48:   # 137 kt
        return 4
    return 5

In [ ]:
def compute_category_kt(vmax_kt): # saffir_simpson_category
    if vmax_kt < 34:
        return -1
    if vmax_kt < 64:
        return 0
    if vmax_kt < 83:
        return 1
    if vmax_kt < 96:
        return 2
    if vmax_kt < 113:
        return 3
    if vmax_kt < 137:
        return 4
    return 5   

In [ ]:
def build_track_data(track_df, event_code, id_no, basin="WP"):
    
    time = pd.to_datetime(track_df[["year", "month", "day", "hour"]]).to_numpy(dtype="datetime64[ns]")

    ds_track = xr.Dataset(coords=dict(time=time,
                                      lat=("time", track_df["lat"].values),
                                      lon=("time", track_df["lon"].values),
                                     ),
                          
                          data_vars=dict(max_sustained_wind=("time", track_df["vmax_ms"].values * 1.94384),
                                         central_pressure=("time", track_df["pmin_hpa"].values),
                                         environmental_pressure=("time", track_df["penv_hpa"].values),
                                         radius_max_wind=("time", np.full(len(time), np.nan)), # see "def estimate_rmw(rmw, cen_pres):"
                                         basin=("time", np.array([basin] * len(time))),
                                         time_step=("time", np.ones(len(time), dtype=float)),
                                        ),

                          attrs=dict(name=event_code[6:], # e.g., "XXXXXXMEKKHALA"
                                     max_sustained_wind_unit="kt",
                                     central_pressure_unit="hPa",
                                     sid=track_df.at[0, "sid"],
                                     id_no=id_no,
                                     orig_event_flag=True,
                                     category=compute_category_ms((track_df["vmax_ms"].values).max()),
                                    ),
                          )
    return ds_track

In [ ]:
###############################################################################################################################################
# preparation for the all_tracks_haz hazard object (without generating probabilistic synthetic events)
###############################################################################################################################################

In [ ]:
all_track_data = []

id_no = 1
for f in tracks_list:
    track_df = pd.read_csv(f,
                           sep=r"\s+",
                           header=None,
                           names=["sid","year","month","day","hour","lon","lat","vmax_ms","pmin_hpa","penv_hpa"]) # pandas.Dataframe

    filename = f.split("/")[-1] # the last element in ["", "lfs", "home", ..., "TCtrack.slp.4C_202002MEKKHALA_48.txt"] 
    event_code = filename.split("_")[1] # the second element in ["TCtrack.slp.4C", "202002MEKKHALA", "48.txt"]

    track_data = build_track_data(track_df, event_code, id_no, basin="WP")
    all_track_data.append(track_data)


    id_no += 1

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0
###############################################################################################################################################
# single_track = TCTracks(data=[all_track_data[-1]])
# single_track.plot().set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree()) # boundary the same as WRF
###############################################################################################################################################
all_tracks = TCTracks(data=all_track_data)
all_tracks.plot().set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree()) # boundary the same as WRF

In [ ]:
###############################################################################################################################################
# below we generate probabilistic/synthetic events
# for the all_tracks_haz hazard object
###############################################################################################################################################

In [ ]:
# all_tracks.equal_timestep() # must be done before sythetic generation 
# all_tracks.calc_perturbed_trajectories(nb_synth_tracks=10) # nb_synth_tracks = how many synthetic tracks is computed for every track
#                                                            # generate synthetic tracks based on directed random walk.
# all_tracks.plot()
# all_tracks.data
# all_tracks.data[-1] # the last synthetic track, notice the value of orig_event_flag and name

In [ ]:
###############################################################################################################################################
# create the all_tracks_haz hazard object (after generating probabilistic synthetic events)
###############################################################################################################################################

In [ ]:
# centroids = from WRF
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()])
###############################################################################################################################################
# other Hazard variables 
# ...
###############################################################################################################################################
# other Hazard variables  
# ...

In [ ]:
all_tracks_haz = TropCyclone.from_tracks(tracks=all_tracks, 
                                         centroids=centroids, # centroids = from WRF
                                         model='H1980', 
                                         model_kwargs={"gradient_to_surface_winds": 0.9},
                                         intensity_thres=0) 

In [ ]:
all_tracks_haz.check()

In [ ]:
all_tracks_haz.plot_intensity(0, vmin=0, vmax=70)
all_tracks_haz.plot_intensity(1)
all_tracks_haz.plot_intensity(2)
all_tracks_haz.plot_intensity(21)
all_tracks_haz.plot_intensity(22)
all_tracks_haz.plot_intensity("202002MEKKHALA", vmin=0, vmax=40)
all_tracks_haz.plot_intensity("202003BAVI", vmin=0, vmax=40)
all_tracks_haz.plot_intensity("202004ATSANI", vmin=0, vmax=40)

In [ ]:
###############################################################################################################################################
# WRF vs. Holland  
# wrf minus holland (by max)
###############################################################################################################################################

In [ ]:
# centroids = from WRF
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()])
###############################################################################################################################################
# intensity by max
diff_by_max = haz.intensity.max(axis=0).toarray().ravel() - all_tracks_haz.intensity.max(axis=0).toarray().ravel()

diff_by_max = sparse.csr_matrix(diff_by_max.reshape(1, -1))

###############################################################################################################################################
# fraction by max
fraction_by_max = diff_by_max.copy()
fraction_by_max.data.fill(1)
###############################################################################################################################################
frequency_by_max = np.array([1.0]) # placeholder for Hazard object
###############################################################################################################################################
event_id_by_max = np.array([1]) # placeholder for Hazard object
###############################################################################################################################################
event_name_by_max = ["max_wrf_minus_max_holland"]
###############################################################################################################################################
# other Hazard variables = the same as WRF 
# ...

In [ ]:
diff_haz = Hazard(intensity=diff_by_max,
                  fraction=fraction_by_max,
                  centroids=centroids,  
                  units=units,
                  frequency=frequency_by_max,  
                  event_id=event_id_by_max,  
                  event_name=event_name_by_max
                 )

In [ ]:
diff_haz.check()

In [ ]:
cmap = plt.get_cmap("RdBu_r")
levels = [-20, -15, -10, -5, 0, 5, 10, 15, 20]
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)
diff_haz.plot_intensity(1, norm=norm, cmap=cmap, smooth=False)

In [ ]:
###############################################################################################################################################
# exceedance_intensities
###############################################################################################################################################

In [ ]:
###############################################################################################################################################
# return periods
###############################################################################################################################################

In [ ]:
###############################################################################################################################################
# ERA5 anomaly for CNN 
###############################################################################################################################################

In [ ]:
wind_anom_tw = xr.open_dataset('/lfs/home/yanlan/climada/tc-risk/ERA5/wind_anom_tw.nc')

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]



pts_lat = xr.DataArray(lat, dims="points")
pts_lon = xr.DataArray(lon, dims="points") 




regrid = [] # regrid[0], ..., regrid[196]
for track in all_track_data:

    track_times = pd.to_datetime(track["time"].values)

    wind_anom_tw_time_slice = wind_anom_tw.sel(time=track_times)

    # max anomaly during track lifetime
    wind_anom_tw_max = wind_anom_tw_time_slice.max(dim="time")

    # sort before interpolation
    wind_anom_tw_max = wind_anom_tw_max.sortby(["latitude", "longitude"])

    # interpolate ERA5 anomaly field to WRF mask points
    r = wind_anom_tw_max.interp(latitude=pts_lat,
                                longitude=pts_lon,
                                method="linear")

    regrid.append(r)

In [ ]:
###############################################################################################################################################
# Terrain for CNN 
###############################################################################################################################################

In [ ]:
terr = xr.open_dataset("/lfs/home/yanlan/climada/tc-risk/terrain/wrf_pgw_twn_grid_coords.nc")

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]




# terr
ter_lat = terr["XLAT"].values.ravel()
ter_lon = terr["XLONG"].values.ravel()
ter_values = terr["TER"].values.ravel()

# interpolation
# r_terrain is a 1d numpy array
r_terrain = griddata(points=(ter_lon, ter_lat), # from terr points
                     values=ter_values,
                     xi=(lon, lat), # to WRF mask points
                     method="linear")



In [ ]:
###############################################################################################################################################
# CNN 197 events <-> 197 events
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of (499*549)


n_ev = all_tracks_haz.intensity.shape[0] # use the input X

y_idx, x_idx = np.where(mask.values)
ny = y_idx.max() - y_idx.min() + 1
nx = x_idx.max() - x_idx.min() + 1
mask_rect = mask.isel( south_north = slice(y_idx.min(), y_idx.max() + 1),
                       west_east   = slice(x_idx.min(), x_idx.max() + 1) ) # mask_rect is of (254*235)

X_img     = np.zeros((n_ev, ny, nx))
era5_img  = np.zeros((n_ev, ny, nx))
terr_img  = np.zeros((n_ev, ny, nx))
Y_img     = np.zeros((n_ev, ny, nx))


X_img[:, mask_rect.values]    = all_tracks_haz.intensity.toarray() 
era5_img[:, mask_rect.values] = np.stack( [r["sqrt_wind_speed_anomaly"].values for r in regrid],
                                          axis=0 ) 
terr_img[:, mask_rect.values] = r_terrain # r_terrain is a 1d numpy array
Y_img[:, mask_rect.values]    = haz.intensity.toarray() 



X_cnn   = np.stack([X_img, era5_img, terr_img], axis=1) # numpy.ndarray (n_ev, channels, ny, nx)
Y_cnn   = np.stack([Y_img          ], axis=1) # numpy.ndarray (n_ev, 1, ny, nx)
res_cnn = Y_cnn - X_cnn[:, 0:1, :, :] # 0:1 selects the channel in X_cnn

In [ ]:
X_tensor   = torch.tensor(X_cnn, dtype=torch.float32)
Y_tensor   = torch.tensor(Y_cnn, dtype=torch.float32)
res_tensor = torch.tensor(res_cnn, dtype=torch.float32)

In [ ]:
torch.manual_seed(42) # fix training, validation, test samples
                      # fix initial CNN weights 
perm           = torch.randperm(n_ev)
train_idx      = perm[:int(0.70 * n_ev)] # e.g., train_idx = torch.tensor([0, 1])  
validation_idx = perm[int(0.70 * n_ev):int(0.85 * n_ev)]
test_idx       = perm[int(0.85 * n_ev):] # e.g., test_idx = torch.tensor([2])  

In [ ]:
X_tensor_train        = X_tensor[train_idx]
X_tensor_validation   = X_tensor[validation_idx]
X_tensor_test         = X_tensor[test_idx]

Y_tensor_test         = Y_tensor[test_idx]

res_tensor_train      = res_tensor[train_idx]
res_tensor_validation = res_tensor[validation_idx]

In [ ]:
X_mean        = X_cnn[train_idx][:, 0:3, mask_rect.values].mean(axis=(0, 2)) # axis=0: n_ev
                                                                             # axis=2: n_mask_points
X_mean        = X_mean.reshape(1, 3, 1, 1) 
X_mean_tensor = torch.tensor(X_mean, dtype=torch.float32)

X_std         = X_cnn[train_idx][:, 0:3, mask_rect.values].std(axis=(0, 2)) 
X_std         = X_std.reshape(1, 3, 1, 1) 
X_std_tensor  = torch.tensor(X_std, dtype=torch.float32)

mask_rect_tensor = torch.tensor(mask_rect.values, dtype=torch.bool)

In [ ]:
class CNN(nn.Module): 
    def __init__(self, X_mean_tensor, X_std_tensor, mask_rect_tensor):
        super().__init__() # nn.Module.__init__(self)

        self.net    = nn.Sequential(nn.Conv2d(3, 16, kernel_size=(3, 3), stride=1, padding=1), # input: (n_hours, channels, nlat, nlon) -> (n_hours, 16, nlat, nlon)                               
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU

                                    nn.Conv2d(16, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 16, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU
 
                                    nn.Conv2d(32, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU

                                    nn.Conv2d(32, 16, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 16, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(16, 1, kernel_size=(3, 3), stride=1, padding=1) # output: res_pred 
                                                                                              # output: (n_hours, 1, nlat, nlon)
                                                                                              # Y_pred = X + res_pred
                                    )

        self.X_mean_tensor    = X_mean_tensor # X_mean is of (1, channels, 1, 1)
        self.X_std_tensor     = X_std_tensor # X_std is of (1, channels, 1, 1)
        self.mask_rect_tensor = mask_rect_tensor # mask_rect_tensor is of (254, 235)


    def forward(self, X):
        
        normalized_X = (X - self.X_mean_tensor) / self.X_std_tensor # X is of torch.tensor
                                                                    # normalized_X is of torch.tensor
        
        normalized_X[:, :, ~self.mask_rect_tensor] = 0.0     
        return self.net(normalized_X) # meaning nn.Sequential(...)(normalized_X)

In [ ]:
model = CNN(X_mean_tensor, X_std_tensor, mask_rect_tensor)

loss_func = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 200

for epoch in range(n_epochs):
    # forward
    model.train()
    r_pred = model(X_tensor_train) # model.__call__(X_tensor) 
                                  # model.forward(X_tensor[train_idx])
    r_true = res_tensor_train
    
    train_loss = loss_func(r_pred[:, :, mask_rect_tensor],  
                           r_true[:, :, mask_rect_tensor])

    # learning weights
    optimizer.zero_grad()
    train_loss.backward() # computes the new gradients
    optimizer.step() # update model weights
    
    if epoch % 10 == 0: 
       # validation
       model.eval()
       with torch.no_grad():
           r_validation_pred = model(X_tensor_validation)
           r_validation_true = res_tensor_validation
           validation_loss = loss_func(r_validation_pred[:, :, mask_rect_tensor], 
                                       r_validation_true[:, :, mask_rect_tensor])


           print(f"epoch {epoch}, "
                 f"train mse: {train_loss.item():.6f}, "
                 f"validation mse: {validation_loss.item():.6f}")

In [ ]:
model.eval()
with torch.no_grad():

    # model predicts residual
    r_pred_test = model(X_tensor_test)

    # final corrected wind
    Y_pred_test = X_tensor_test[:, 0:1, :, :] + r_pred_test

    mae = torch.mean( torch.abs(Y_tensor_test[:, :, mask_rect_tensor] - 
                                Y_pred_test[:, :, mask_rect_tensor]) )
    
    rmse = torch.sqrt( torch.mean((Y_tensor_test[:, :, mask_rect_tensor] -
                                   Y_pred_test[:, :, mask_rect_tensor]) ** 2) )

    old_mae = torch.mean(torch.abs(Y_tensor_test[:, :, mask_rect_tensor] - 
                                   X_tensor_test[:, 0:1, mask_rect_tensor]) )
    
    old_rmse = torch.sqrt(torch.mean((Y_tensor_test[:, :, mask_rect_tensor] - 
                                      X_tensor_test[:, 0:1, mask_rect_tensor]) ** 2) )


In [ ]:
print("old MAE:", old_mae.item())
print("old RMSE:", old_rmse.item())
print("MAE:", mae.item())
print("RMSE:", rmse.item())

In [ ]:
###############################################################################################################################################
# CNN 197 events <-> 197 events
# TESTING VISUALIZATION
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of xarray.DataArray

    centroids = Centroids( lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                           lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()] )

###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
Y_pred_test_n_ev = len(Y_pred_test)
###############################################################################################################################################
Y_pred_test_event_id = test_idx.numpy()+1    
###############################################################################################################################################
# from (n_events, 1, nlat, nlon) to (n_events, nlat, nlon)
Y_pred_test_np = Y_pred_test.squeeze(1).cpu().numpy() 

# from (n_events, nlat, nlon) to (n_events, n_centroids)
flat_Y_pred_test = Y_pred_test_np[:, mask_rect.values] 

# # turn negative to positive for the compressed sparse matrix
# flat_Y_pred_test = np.maximum(flat_Y_pred_test, 0)

# convert to intensity sparse matrix
Y_pred_test_intensity = sparse.csr_matrix(flat_Y_pred_test.astype(np.float32))
###############################################################################################################################################
Y_pred_test_frequency =  np.ones(Y_pred_test_n_ev) # placeholder for Hazard object  
###############################################################################################################################################
Y_pred_test_fraction = Y_pred_test_intensity.copy()
Y_pred_test_fraction.data.fill(1)
###############################################################################################################################################
Y_pred_test_event_name = [all_tracks_haz.event_name[i] for i in test_idx.numpy()] 
###############################################################################################################################################
# date = np.array([  ])
###############################################################################################################################################
# orig = np.ones(n_ev, bool)

In [ ]:
Y_pred_test_haz = Hazard(intensity=Y_pred_test_intensity,
                         fraction=Y_pred_test_fraction,
                         centroids=centroids,  
                         units=units,
                         frequency=Y_pred_test_frequency,  
                         event_id=Y_pred_test_event_id,  
                         event_name=Y_pred_test_event_name)

In [ ]:
Y_pred_test_haz.check()

In [ ]:
Y_pred_test_haz.event_id

In [ ]:
ax = all_tracks_haz.plot_intensity(event=96, vmin=0, vmax=60) # bar's true top = your data max
cbar_ax = ax.get_figure().axes[-1]
cbar_ax.set_yticks([0, 15, 30, 45]) 


ax = Y_pred_test_haz.plot_intensity(event=96, vmin=0, vmax=60) # bar's true top = your data max
cbar_ax = ax.get_figure().axes[-1]
cbar_ax.set_yticks([0, 15, 30, 45]) 


ax = haz.plot_intensity(event=96, vmin=0, vmax=60) # bar's true top = your data max
cbar_ax = ax.get_figure().axes[-1]
cbar_ax.set_yticks([0, 15, 30, 45]) 

